In [117]:
# importing required libraries
import pandas as pd
import random

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem.wordnet import WordNetLemmatizer

from gensim.corpora import Dictionary

In [94]:
# Dataset
data = pd.read_csv("cyberbullying.csv")
data = data.dropna()  # Drop rows with missing values
data = data.drop_duplicates()  # Drop duplicate rows

print(data.head(5))

                                          tweet_text cyberbullying_type
0  In other words #katandandre, your food was cra...  not_cyberbullying
1  Why is #aussietv so white? #MKR #theblock #ImA...  not_cyberbullying
2  @XochitlSuckkks a classy whore? Or more red ve...  not_cyberbullying
3  @Jason_Gio meh. :P  thanks for the heads up, b...  not_cyberbullying
4  @RudhoeEnglish This is an ISIS account pretend...  not_cyberbullying


In [95]:
# Negation Code
# from https://github.com/UtkarshRedd/Negation_handling

def negation_handler(sentence):
    temp = int(0)
    for i in range(len(sentence)):
        if sentence[i-1][-3:] in ["n't"]: # Minor edit here to always take last 3 characters for the if statement
            antonyms = []
            for syn in wordnet.synsets(sentence[i]):
                syns = wordnet.synsets(sentence[i])
                w1 = syns[0].name()
                temp = 0
                for l in syn.lemmas():
                    if l.antonyms():
                        antonyms.append(l.antonyms()[0].name())
                max_dissimilarity = 0
                for ant in antonyms:
                    syns = wordnet.synsets(ant)
                    w2 = syns[0].name()
                    syns = wordnet.synsets(sentence[i])
                    w1 = syns[0].name()
                    word1 = wordnet.synset(w1)
                    word2 = wordnet.synset(w2)
                    if isinstance(word1.wup_similarity(word2), float) or isinstance(word1.wup_similarity(word2), int):
                        temp = 1 - word1.wup_similarity(word2)
                    if temp>max_dissimilarity:
                        max_dissimilarity = temp
                        antonym_max = ant
                        sentence[i] = antonym_max
                        sentence[i-1] = ''
    while '' in sentence:
        sentence.remove('')
    return sentence

In [124]:
# Preprocessing Data code
# Self-written

def data_cleaning(data):
    for entry in data.tweet_text:
        # Lowercase
        lower = entry.lower()

        # Negation handling
        splits = lower.split()
        negated = negation_handler(splits)
        words = ' '.join(negated)

        # Tokenization
        tokenizer = RegexpTokenizer(r'\w+')
        tokens = tokenizer.tokenize(words)

        # Remove stop words
        stop_words = set(stopwords.words('english'))
        stopped = [word for word in tokens if word not in stop_words]

        # Lemmatize
        lemmatizer = WordNetLemmatizer()
        lemmatize = [lemmatizer.lemmatize(word) for word in stopped]

        # Join words back into a single string
        cleaned_entry = ' '.join(lemmatize)

        # Update the DataFrame with the cleaned entry and lemmatized words
        data.loc[data.tweet_text == entry, 'cleaned_words'] = cleaned_entry


In [125]:
# Clean the data
data_cleaning(data)
print(data.head(5))

                                          tweet_text cyberbullying_type  \
0  In other words #katandandre, your food was cra...  not_cyberbullying   
1  Why is #aussietv so white? #MKR #theblock #ImA...  not_cyberbullying   
2  @XochitlSuckkks a classy whore? Or more red ve...  not_cyberbullying   
3  @Jason_Gio meh. :P  thanks for the heads up, b...  not_cyberbullying   
4  @RudhoeEnglish This is an ISIS account pretend...  not_cyberbullying   

                                       cleaned_words  lemmatized_words  
0             word katandandre food crapilicious mkr               NaN  
1  aussietv white mkr theblock imacelebrityau tod...               NaN  
2     xochitlsuckkks classy whore red velvet cupcake               NaN  
3  jason_gio meh p thanks head concerned another ...               NaN  
4  rudhoeenglish isi account pretending kurdish a...               NaN  


In [126]:
# Check for empty cleaned words and do something????
data[data['cleaned_words']=='']

,tweet_text,cyberbullying_type,cleaned_words,lemmatized_words
307,:D,not_cyberbullying,,NaN
2675,That is all,not_cyberbullying,,NaN
4011,♪♥♪,not_cyberbullying,,NaN
4016,#M…,not_cyberbullying,,NaN
4582,Why?,not_cyberbullying,,NaN
6451,No. Just no.,not_cyberbullying,,NaN
6545,Just NOW?!?!? 😄😃😀,not_cyberbullying,,NaN
8695,😂😂😂😂😂,gender,,NaN
9981,👦 no it isn’t,gender,,NaN
14458,👧👧👧👧 …,gender,,NaN


In [130]:
# Create a dictionary representation
dictionary = Dictionary(data['cleaned_words'].apply(lambda x: x.split()))

# Transform the cleaned words into a bag-of-words representation
corpus = [dictionary.doc2bow(tweet.split()) for tweet in data['cleaned_words']]

print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 56723
Number of documents: 47656
